In [5]:
import argparse
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM, 
    AutoTokenizer, 
    Trainer, 
    TrainingArguments, 
    DataCollatorForLanguageModeling
)

In [8]:
# Add parameter parsing to control whether to train the original version or the stretch version
parser = argparse.ArgumentParser()
parser.add_argument('--variant', type=str, default='original', choices=['original', 'reduced'], help="Model variant to train")
# current_args = [] # original
current_args = ['--variant', 'reduced'] # reduced 
args = parser.parse_args(args=current_args)

TRAIN_FILE = "data/train.json"
VAL_FILE = "data/val.json"
MODEL_NAME = "gpt2"

In [9]:
def main():

    # Load dataset
    dataset = load_dataset("json", data_files={"train": TRAIN_FILE, "validation": VAL_FILE})

    # Tokenizer
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    
    # GPT-2 doesn't have a pad token, so we use eos_token
    tokenizer.pad_token = tokenizer.eos_token

    def tokenize_function(examples):
        return tokenizer(examples["text"], truncation=True, max_length=512)

    tokenized_datasets = dataset.map(tokenize_function, batched=True, remove_columns=["text"])

    # Data collator (Handles dynamic padding)
    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

    # Model
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

    
    # Main modification: in reduced mode, remove one transformer layer. 
    if args.variant == 'reduced':
        print("Performing truncation: Removing the last Transformer layer...")
        
        # The layers of GPT-2 are stored in model.transformer.h.
        # We retrieve the existing layer list, remove the last entry, and repackage it into a ModuleList.
        # Original: 12 layers -> New: 11 layers
        current_layers = list(model.transformer.h)
        model.transformer.h = torch.nn.ModuleList(current_layers[:-1])
        
        # Update the configuration; otherwise, errors will occur during saving/loading.
        model.config.n_layer = model.config.n_layer - 1
        print(f"New model configuration: {model.config.n_layer} layers.")

    # Define different output directories to prevent overwriting.
    OUTPUT_DIR = f"./models/bmw-gpt2-reduced/{args.variant}_model"

    # The same setting as Assignment 1
    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        overwrite_output_dir=True,
        num_train_epochs=5, 
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2, 
        eval_strategy="steps",
        eval_steps=5,                   # Evaluate frequently to show logs
        save_steps=10,
        learning_rate=5e-5,
        weight_decay=0.01,
        logging_dir=f'./logs/{args.variant}', # different logs
        logging_steps=5,
        use_cpu=not torch.cuda.is_available(), # Fallback if no GPU
        report_to="none"                       # Disable wandb for simple local demo
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_datasets["train"],
        eval_dataset=tokenized_datasets["validation"],
        data_collator=data_collator,
    )

    print(f"Starting training for variant: {args.variant}")
    trainer.train()
    
    # Save model
    print("Saving model...")
    trainer.save_model(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
    print(f"Finished. Model saved to {OUTPUT_DIR}")

if __name__ == "__main__":
    main()

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Performing truncation: Removing the last Transformer layer...
New model configuration: 11 layers.
Starting training for variant: reduced


Step,Training Loss,Validation Loss
5,4.560300,4.085310
10,3.968700,3.669255
15,3.495100,3.500201
20,3.262600,3.353709
25,3.068100,3.265172
30,3.093300,3.225045
35,2.693300,3.181089
40,2.822800,3.167091
45,2.690700,3.165378


Saving model...
Finished. Model saved to ./models/bmw-gpt2-reduced/reduced_model
